In [1]:
import json
import re
import pandas as pd

In [2]:
# Define paths to the downloaded dataset files
BUSINESS_JSON_PATH = "../data/yelp_academic_dataset_business.json"
REVIEW_JSON_PATH = "../data/yelp_academic_dataset_review.json"
OUTPUT_CSV_PATH = "../data/processed_allergy_hazard_dataset.csv"

**Step 1:** Filtering Food & Restaurant Businesses

In [3]:
# Extract a high-lookup set of food and restaurant business IDs
eligible_business_ids = set()
with open(BUSINESS_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        biz = json.loads(line)
        categories = biz.get("categories")
        if categories and ("Restaurants" in categories or "Food" in categories):
            eligible_business_ids.add(biz["business_id"])

print(f"Found {len(eligible_business_ids)} food/restaurant businesses.")

Found 64616 food/restaurant businesses.


**Step 2:** Streaming and Filtering Reviews

In [4]:
# Compile regular expression for target keyword matching
allergy_keywords = re.compile(
    r"\b(allergy|allergic|celiac|anaphylactic|epipen|epi-pen|cross-contamination|glutened|contamination)\b",
    re.IGNORECASE,
)

processed_reviews = []
target_sample_size = 20000  # Sets an optimal sample size for high-caliber modeling

with open(REVIEW_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        review = json.loads(line)

        # 1. Record Filtering: Ensure the review belongs to our target restaurant subset
        if review["business_id"] not in eligible_business_ids:
            continue

        text = review.get("text", "")

        # 2. Outlier Correction: Filter out extreme lengths (spam or text truncation)
        word_count = len(text.split())
        if word_count < 5 or word_count > 800:
            continue

        # 3. Check for target keyword inclusion
        is_allergy_related = bool(allergy_keywords.search(text))

        # Establish initial supervised dependent variable proxy:
        # Reviews containing severe keywords combined with low star ratings are labeled 1, else 0
        is_hazard = 1 if (is_allergy_related and review["stars"] <= 3) else 0

        # 4. Other Preprocessing: Whitespace Normalization
        cleaned_text = text.replace("\n", " ").replace("\t", " ")
        cleaned_text = re.sub(r"\s+", " ", cleaned_text).strip()

        # 5. Feature Engineering: Create non-textual tabular metadata variables
        char_count = len(cleaned_text)
        exclamation_count = cleaned_text.count("!")

        # 6. Variable Filtering: Retain only modeled dimensions (Drop high-cardinality IDs)
        review_features = {
            "stars": review["stars"],
            "useful": review["useful"],
            "funny": review["funny"],
            "cool": review["cool"],
            "text": cleaned_text,
            "word_count": word_count,
            "char_count": char_count,
            "exclamation_count": exclamation_count,
            "is_hazard": is_hazard,
        }

        processed_reviews.append(review_features)

        # Break stream once cap size is achieved to respect 3-week computational limits
        if len(processed_reviews) >= target_sample_size:
            break

print(f"Successfully processed and collected {len(processed_reviews)} reviews.")

Successfully processed and collected 20000 reviews.


**Step 3:** Saving Processed Dataset to CSV

In [5]:
# Convert list of dicts to Pandas DataFrame
df = pd.DataFrame(processed_reviews)

# Log schema layout and class distribution details to verify balance
print(df.info())
print("\nClass distribution of the dependent variable (is_hazard):")
print(df["is_hazard"].value_counts())

# Export the structured dataframe to disk
df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")
print(f"\nDataset successfully saved to: {OUTPUT_CSV_PATH}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   stars              20000 non-null  float64
 1   useful             20000 non-null  int64  
 2   funny              20000 non-null  int64  
 3   cool               20000 non-null  int64  
 4   text               20000 non-null  object 
 5   word_count         20000 non-null  int64  
 6   char_count         20000 non-null  int64  
 7   exclamation_count  20000 non-null  int64  
 8   is_hazard          20000 non-null  int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 1.4+ MB
None

Class distribution of the dependent variable (is_hazard):
is_hazard
0    19976
1       24
Name: count, dtype: int64

Dataset successfully saved to: ../data/processed_allergy_hazard_dataset.csv


In [8]:
df.head(n=100)  # Display the first few rows of the processed dataset for verification

,stars,useful,funny,cool,text,word_count,char_count,exclamation_count,is_hazard
0,3.0,0,0,0,"If you decide to eat here, just be aware it is...",101,511,1,0
1,3.0,0,0,0,Family diner. Had the buffet. Eclectic assortm...,55,339,0,0
2,5.0,1,0,1,"Wow! Yummy, different, delicious. Our favorite...",40,235,6,0
3,4.0,1,0,1,Cute interior and owner (?) gave us tour of up...,94,534,1,0
4,1.0,1,2,1,I am a long term frequent customer of this est...,65,341,2,0
...,...,...,...,...,...,...,...,...,...
95,5.0,2,0,1,Milktooth is the place to go if you want a goo...,183,956,1,0
96,5.0,1,0,0,Not sure why it took until now for us to find ...,100,556,1,0
97,5.0,1,0,0,"Fabulous! Best happy hour, fresh menu, Delish-...",18,98,2,0
98,5.0,0,0,0,My husband and I come here often. We love the ...,32,168,0,0
